<a href="https://colab.research.google.com/github/Cavalheiro93/mvp-machine-learning-spotify-tracks/blob/main/projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://m.media-amazon.com/images/I/31B2Nyzd8XL.png" width="95"/>
<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT1FuUUPNvO7U6WaQX6z35mVyznOojg1JptbQ&s
" width="95"/>
<img src="https://m.media-amazon.com/images/I/31B2Nyzd8XL.png" width="95"/>
<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT1FuUUPNvO7U6WaQX6z35mVyznOojg1JptbQ&s
" width="95"/>
<img src="https://m.media-amazon.com/images/I/31B2Nyzd8XL.png" width="95"/>
<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT1FuUUPNvO7U6WaQX6z35mVyznOojg1JptbQ&s
" width="95"/>
<img src="https://m.media-amazon.com/images/I/31B2Nyzd8XL.png" width="95"/>
<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT1FuUUPNvO7U6WaQX6z35mVyznOojg1JptbQ&s
" width="95"/>


# Projeto Machine Learning | Spotify Tracks

## 00. Sumário & Setup  

- [01. Contexto & Definição do Problema](#01-contexto--definicao-do-problema)  
- [02. Dicionário de Dados](#02-dicionario-de-dados)  


## 01. Contexto & Definição do Problema

O dataset utilizado contém milhares de músicas de diversos gêneros presentes no Spotify. Para este projeto, filtramos apenas as faixas de rock, que englobam diferentes subgêneros como rock alternativo, grunge, punk rock, emo, etc...

Nosso objetivo é, a partir das features numéricas fornecidas pelo Spotify (ex.: danceability, energy, valence, loudness, tempo...), prever não apenas o subgênero principal em que cada faixa mais se encaixa, mas também identificar outros subgêneros secundários que ela possa possuir, construído a partir de modelos de classificação supervisionada capazes de identificar o estilo em que a música melhor se enquadra.

## 02. Dicionario de Dados

| **Recurso de Áudio** | **Descrição**                                            | **Tipo de Dado** |
| -------------------- | -------------------------------------------------------- | ---------------- |
| `track_id`           | Id ou código da faixa                                    | String           |
| `artists`            | Nome do artista ou banda da faixa                        | String           |
| `album_name`         | Nome do álbum da faixa                                   | String           |
| `track_name`         | Nome da faixa musical                                    | String           |
| `popularidade`       | Popularidade da faixa (0 a 100)                          | Int              |
| `duration_ms`        | Duração em milissegundos da faixa                        | Int              |
| `explicit`           | Binário, se a faixa é explícita (contém palavrão ou não) | Bool (0 ou 1)     |
| `danceability`       | Adequação da faixa para dança (0.0 a 1.0)                | Float            |
| `energy`             | Medida perceptiva de intensidade e atividade (0.0 a 1.0) | Float            |
| `key`                | Tom da faixa, varia de -1 a -11                          | Int              |
| `loudness`           | Volume geral da faixa em decibéis (-60 a 0 dB)           | Float            |
| `mode`               | Modalidade da faixa (Maior = 1 / Menor = 0)              | Int (0 ou 1)     |
| `speechiness`        | Presença de palavras faladas na faixa (0.0 a 1.0)        | Float            |
| `acousticness`       | Confiança de que a faixa é acústica (0.0 a 1.0)          | Float            |
| `instrumentalness`   | Indica se a faixa contém vocais (0.0 a 1.0)              | Float            |
| `liveness`           | Presença de público na gravação (0.0 a 1.0)              | Float            |
| `valence`            | Positividade musical (0.0 a 1.0)                         | Float            |
| `tempo`              | Tempo da faixa em batidas por minuto (BPM)               | Float            |
| `time_signature`     | Compasso estimado (3 a 7)                                | Int              |
| `track_genre`        | O gênero ao qual a faixa pertence                        | String           |


## 03. Carga e preparação dos dados

In [ ]:
# configuração para não exibir os warnings
import warnings
warnings.filterwarnings("ignore")

import kagglehub
import os
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier


In [ ]:
# Carrega o dataset a partir da URL especificada, desativando a otimização de memória
# df = pd.read_csv("hf://datasets/maharshipandya/spotify-tracks-dataset/dataset.csv", low_memory=False)

# URL do arquivo Excel no GitHub
github_url = "https://github.com/Cavalheiro93/mvp-machine-learning-spotify-tracks/blob/58b14654e21915b48c64f06f1aee406b786e71da/Spotify_tracks_genre_updated.xlsx"

# Para ler arquivos do GitHub, precisamos da URL "raw"
raw_url = github_url.replace("blob", "raw")

# Lê o arquivo Excel em um DataFrame pandas
df = pd.read_excel(raw_url)

# 04. Tratamento e limpeza dos dados:

In [ ]:


# Filtro das colunas que usaremos somente para o projeto
usecols = ['artist_name', 'track_name', 'track_id', 'popularity', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# O estudo será baseado apenas em gêneros de rock, listando os subgêneros relevantes
genre_rock = ['alt-rock', 'black-metal', 'death-metal', 'emo', 'hard-rock', 'hardcore', 'heavy-metal', 'metal', 'metalcore',
              'goth', 'psych-rock', 'punk', 'punk-rock', 'rock', 'rock-n-roll', 'grunge', 'alternative', 'industrial',]


# Renomeia a coluna 'track_genre' para 'genre' e 'artists' para 'artist_name' para padronização
df = df.rename(columns={'track_genre': 'genre', 'artists': 'artist_name'})

# Seleciona apenas as colunas definidas em usecols
df = df[usecols]

# Filtra o DataFrame para incluir apenas os gêneros presentes na lista genre_rock
# filtro_rock = df['genre'].isin(genre_rock)

# Filtra músicas com popularidade maior que 1 para remover faixas muito desconhecidas
filtro_popularidade = df['popularity'] > 1

# Filtra músicas com speechiness diferente de 0 (para incluir faixas com alguma fala)
filtro_speechiness = df['speechiness'] != 0
# Drop na coluna 'artist_name' onde houver os artistas abaixo
lista_artists = ['XXXTENTACION', '6ix9ine', 'Big L', '916frosty', '6 Dogs', '50landing', 'Big D', 'NEZS Beats', 'Big Boy'
                  '24SJU', 'Adoo']
filtro_artistas = df['artist_name'].str.contains('|'.join(lista_artists)) == False


# Aplica os filtros ao DataFrame
# df = df[filtro_rock]
df = df[filtro_popularidade]
df = df[filtro_speechiness]
df = df[filtro_artistas]

In [ ]:
# Garantir nomes limpos (opcional)
df['artist_name'] = df['artist_name'].astype(str).str.strip()

# 1) Tabela de contagens por artista x gênero
tabela = pd.crosstab(df['artist_name'], df['genre'])

# 2) Reorganiza colunas na ordem de genre_rock e preenche ausências com 0
tabela = tabela.reindex(columns=genre_rock, fill_value=0)

# 3) Gênero mais frequente por artista
# Obs.: em caso de empate, idxmax pega o primeiro na ordem de 'genre_rock'
genre_grouped = tabela.idxmax(axis=1)

# 4) Resultado final
genre_grouped = tabela.assign(genre_grouped=genre_grouped).reset_index()

In [ ]:
# Substitui a coluna 'genre' pelo agrupado
df = df.drop(columns=['genre'])  # remove o antigo
df = df.merge(genre_grouped[['artist_name', 'genre_grouped']],
              on='artist_name', how='left')

# Renomeia para manter só 'genre'
df = df.rename(columns={'genre_grouped': 'genre'})

mapa_generos = {
    "punk": "punk",
    "punk-rock": "punk",
    "rock": "alternative rock",
    "alt-rock": "alternative rock",
    "alternative": "alternative rock",
    "grunge": "alternative rock",
    "heavy-metal": "heavy-metal and goth",
    "goth": "heavy-metal and goth",
    "hard-rock": "heavy-metal and goth",
    "metal": "new metal",
    "metalcore": "new metal",
    "industrial": "new metal",
    "garage": "alternative rock",
}

# aplicar o mapeamento
df["genre_grouped"] = df["genre"].map(lambda g: mapa_generos.get(g, g))

# remover a coluna antiga de gênero
df.drop(columns=['genre'], inplace=True)

# ordenar por popularidade decrescente
df = df.sort_values(by='popularity', ascending=False)

#remover duplicadas onde o track_id é igual, mantendo a mais popular
df = df.drop_duplicates(subset=['track_id'], keep='first')

In [ ]:
# Removeção do grupo "outros"
df = df.copy()

# Somente colunas para o modelo
df_ml = df.drop(columns=["key", "mode", "artist_name", "track_name", "track_id", 'popularity',
                         'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', ])

# Somente colunas para o modelo, SEGUNDA TENTATIVA
df_ml = df.drop(columns=["key", "mode", "artist_name", "track_name", "track_id", 'speechiness', 'liveness'])

# Somente colunas para o modelo, TERCEIRA TENTATIVA
#df_ml = df.drop(columns=["key", "mode", "artist_name", "track_name", "track_id", 'speechiness', 'liveness', 'acousticness', 'instrumentalness'])

## 05. Análise Exploratória (EDA)

#### Dimensão do Dataset

In [ ]:
# Após todos os tratamentos iniciais, temos a dimensão final do dataset
df_ml.shape

In [ ]:
df_ml.info()

- *não temos valores nulos em nenhuma das colunas*
- *No dataset geral temos 3 tipos de variaveis, sendo object, int e float*

#### Estatísticas Descritivas das variáveis numéricas

In [ ]:
df_ml.describe().T

- **Popularity**: ampla variação entre músicas pouco e muito populares (mín. 2, máx. 96), média em torno de 42.  
- **Danceability**: distribuição equilibrada (média ~0.47), variando de músicas pouco dançantes a bastante dançantes.  
- **Energy**: valores geralmente altos (média ~0.76), confirmando que a maioria das faixas tem perfil energético típico do rock.  
- **Loudness**: média em torno de -6.6 dB, mas com grande variação (-29 até +1), indicando diferenças relevantes de intensidade sonora.  
- **Valence**: média ~0.43, mostrando equilíbrio entre faixas mais “alegres” e mais “melancólicas”, com grande dispersão (0.02 a 0.99).  
- **Tempo (BPM)**: média ~126 bpm, mas com músicas desde muito lentas (34 bpm) até muito rápidas (215 bpm).  


#### Mini gráficos para visualização de cada campo

In [ ]:
from matplotlib import pyplot as plt

# 1) Lista automática das colunas numéricas (não pega as categóricas nem o target)
numeric_cols = df_ml.select_dtypes(include="number").columns.tolist()

# 2) Grid dinâmico (2 colunas) – funciona com qualquer quantidade de variáveis
n = len(numeric_cols)
rows = (n + 1) // 2  # arredonda pra cima
fig, axes = plt.subplots(nrows=rows, ncols=2, figsize=(12, 3*rows))
axes = axes.ravel()

# 3) Plota os histogramas
for i, col in enumerate(numeric_cols):
    axes[i].hist(df_ml[col].dropna(), bins=30)
    axes[i].set_title(col)
    axes[i].set_xlabel("Valor")
    axes[i].set_ylabel("Frequência")

# 4) Se sobrar “ax” vazio (quando n for ímpar), remove do layout
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


Overview dos gráficos
- *Popularity* tem boa distribuição, entretanto as faixas dificilmente passam dos 80, afirmando que o rock não é tão popular nos dias de hoje.
- Maior concentração dos valores de *danceability* no centro do gráfico, entre os valores 0.4 e 0.6, indicando uma grande variedade de estilos de música dentro do mesmo gênero.
- *energy* e *loudness* com maior parte dos seus valores na extremidade da direita. Pode indicar que faixas de rock sejam mais enérgicas e geralmente com volume bem alto.
- Outros campos como *acousticness* e *instrumentalness*, temos grande concentração dos valores proximos de 0.
- No campo *valence* temos uma grande dispersão entre valores 0 e 1, com maior concentração em 0.2 a 0.5, indicando que temos uma grande mistura de estilos musicais.
- No campo *tempo* temos maior concentração entre 50 e 200 BPM. No gênero rock poucas musicas tem ritmo muito lento ou rápido demais.

In [ ]:
# 1) Seleciona apenas features numéricas (ignora 'genre_grouped')
num_cols = df_ml.select_dtypes(include="number").columns
corr = df_ml[num_cols].corr(method="spearman")  # use "pearson" se preferir

# 2) Heatmap (triângulo inferior, com anotações)
mask = np.triu(np.ones_like(corr, dtype=bool))  # esconde triângulo superior
plt.figure(figsize=(8,6))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, square=True,
            cbar_kws={"shrink": .8})
plt.title("Correlação (Spearman) entre features")
plt.tight_layout()
plt.show()

Overview dos gráficos
- *energy ↔ loudness*: Forte correlação (+0.69), trilha mais alta há mais energia
- *valence ↔ danceability*: Forte correlação (+0.48), trilha mais dançante é mais feliz
- *acousticness ↔ danceability*: Correlação forte (+0.42) muito por conta do estilo musical de rock-n-roll e psych-rock
- *acousticness ↔ energy*: Há forte correlação negativa (-0.68) faixas mais acústicas tendem a apresentar menor energia
- *acousticness ↔ loudness*: Moderada correlação negativa (-0.44), faixas mais acústicas tendem a exibir menor nível sonoro

## 06. Divisão  dos  dados  adequada  ao  problema

### Separação em conjunto de treino e conjunto de teste com holdout

In [ ]:
test_size = 0.20 # tamanho do conjunto de teste
seed = 7 # semente aleatória

# Features = tudo menos o alvo
X = df_ml.drop(columns=["genre_grouped"])
# Alvo = a coluna correta
y = df_ml["genre_grouped"].astype(str)  # garante string

X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size=test_size, shuffle=True, random_state=seed, stratify=y) # holdout com estratificação

# Parâmetros e partições da validação cruzada
scoring = 'accuracy'
num_particoes = 10
kfold = StratifiedKFold(n_splits=num_particoes, shuffle=True, random_state=seed) # validação cruzada com estratificação

## 07. Modelagem

### Criação e avaliação de modelos: linha base, dados padronizados e normalizados

In [ ]:
np.random.seed(7) # definindo uma semente global para este bloco

# Listas para armazenar os armazenar os pipelines e os resultados para todas as visões do dataset
pipelines = []
results = []
names = []

# Definindo os parâmetros do classificador base para o BaggingClassifier
base = DecisionTreeClassifier()
num_trees = 100
max_features = 3

# Criando os elementos do pipeline

# Algoritmos que serão utilizados
reg_log = ('LR', LogisticRegression(max_iter=200))
knn = ('KNN', KNeighborsClassifier())
cart = ('CART', DecisionTreeClassifier())
naive_bayes = ('NB', GaussianNB())
svm = ('SVM', SVC())
bagging = ('Bag', BaggingClassifier(estimator=base, n_estimators=num_trees))
random_forest = ('RF', RandomForestClassifier(n_estimators=num_trees, max_features=max_features))
extra_trees = ('ET', ExtraTreesClassifier(n_estimators=num_trees, max_features=max_features))
adaboost = ('Ada', AdaBoostClassifier(n_estimators=num_trees))
gradient_boosting = ('GB', GradientBoostingClassifier(n_estimators=num_trees))


# Defina a lista de estimadores do Voting (não pode ser vazia)
bases = [
    ('lr', LogisticRegression(max_iter=200)),
    ('knn', KNeighborsClassifier()),
    ('cart', DecisionTreeClassifier()),
    ('nb', GaussianNB()),
    ('svm', SVC())
]

# Voting "hard"
voting = ('Voting', VotingClassifier(estimators=bases, voting='hard', n_jobs=-1))


# Transformações que serão utilizadas
standard_scaler = ('StandardScaler', StandardScaler())
min_max_scaler = ('MinMaxScaler', MinMaxScaler())


# Montando os pipelines

# Dataset original
pipelines.append(('LR-orig', Pipeline([reg_log])))
pipelines.append(('KNN-orig', Pipeline([knn])))
pipelines.append(('CART-orig', Pipeline([cart])))
pipelines.append(('NB-orig', Pipeline([naive_bayes])))
pipelines.append(('SVM-orig', Pipeline([svm])))
pipelines.append(('Bag-orig', Pipeline([bagging])))
pipelines.append(('RF-orig', Pipeline([random_forest])))
pipelines.append(('ET-orig', Pipeline([extra_trees])))
pipelines.append(('Ada-orig', Pipeline([adaboost])))
pipelines.append(('GB-orig', Pipeline([gradient_boosting])))
pipelines.append(('Vot-orig', Pipeline([voting])))

# Dataset Padronizado
pipelines.append(('LR-padr', Pipeline([standard_scaler, reg_log])))
pipelines.append(('KNN-padr', Pipeline([standard_scaler, knn])))
pipelines.append(('CART-padr', Pipeline([standard_scaler, cart])))
pipelines.append(('NB-padr', Pipeline([standard_scaler, naive_bayes])))
pipelines.append(('SVM-padr', Pipeline([standard_scaler, svm])))
pipelines.append(('Bag-padr', Pipeline([standard_scaler, bagging])))
pipelines.append(('RF-padr', Pipeline([standard_scaler, random_forest])))
pipelines.append(('ET-padr', Pipeline([standard_scaler, extra_trees])))
pipelines.append(('Ada-padr', Pipeline([standard_scaler, adaboost])))
pipelines.append(('GB-padr', Pipeline([standard_scaler, gradient_boosting])))
pipelines.append(('Vot-padr', Pipeline([standard_scaler, voting])))

# Dataset Normalizado
pipelines.append(('LR-norm', Pipeline([min_max_scaler, reg_log])))
pipelines.append(('KNN-norm', Pipeline([min_max_scaler, knn])))
pipelines.append(('CART-norm', Pipeline([min_max_scaler, cart])))
pipelines.append(('NB-norm', Pipeline([min_max_scaler, naive_bayes])))
pipelines.append(('SVM-norm', Pipeline([min_max_scaler, svm])))
pipelines.append(('Bag-norm', Pipeline([min_max_scaler, bagging])))
pipelines.append(('RF-norm', Pipeline([min_max_scaler, random_forest])))
pipelines.append(('ET-norm', Pipeline([min_max_scaler, extra_trees])))
pipelines.append(('Ada-norm', Pipeline([min_max_scaler, adaboost])))
pipelines.append(('GB-norm', Pipeline([min_max_scaler, gradient_boosting])))
pipelines.append(('Vot-norm', Pipeline([min_max_scaler, voting])))

# Executando os pipelines
for name, model in pipelines:
    cv_results = cross_val_score(model, X_train, y_train, cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    msg = "%s: %.3f (%.3f)" % (name, cv_results.mean(), cv_results.std()) # formatando para 3 casas decimais
    print(msg)

# Boxplot de comparação dos modelos
fig = plt.figure(figsize=(25,6))
fig.suptitle('Comparação dos Modelos - Dataset orginal, padronizado e normalizado')
ax = fig.add_subplot(111)
plt.boxplot(results)
ax.set_xticklabels(names, rotation=90)
plt.show()

Nessa etapa, visualizamos que os modelos que tiveram o melhor desempenho foram principalmente:
- Bagging
- Random Forest
- Extra Trees
- Gradient Boosting

Todos acima dos 0.5.

Conclusões importantes dos resultados:

- Modelos Lineares e probabilísticos simples como **SVM**, **BN** e **LR** não respondem bem ao nosso dataset, para captura dos subgêneros musicais do rock.

- Entretanto, quando aplicamos tanto Padronização e Normalização, percebemos uma melhora significativa nesses resultados, a exemplo do SVM que saiu de 0.274 para 0.472 padronizado e 0.451 normalizado

- Modelos baseados em árvores + ensembles respondem melhor ao dataset.

# 08. Otimização de hiperparâmetros

### Random Forest Refinado


In [ ]:
# K-Fold reduzido para acelerar
kfold = KFold(n_splits=5, shuffle=True, random_state=7)

# --------------------------------------------------------------
# Random Forest Refinado
# --------------------------------------------------------------
pipe_rf = Pipeline([('rf', RandomForestClassifier(random_state=7))])

rf_space = {
    'rf__n_estimators': [200, 400, 600],
    'rf__max_depth': [None, 10, 10],
    'rf__max_features': ['sqrt', 0.5, 0.4],
    'rf__min_samples_leaf': [1, 2, 2],
    'rf__min_samples_split': [2, 4, 6],
    'rf__bootstrap': [True, True, True],
}
rf_search = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=rf_space,
    n_iter=20,             # aumente para cobrir melhor o espaço
    cv=kfold,
    scoring='accuracy',
    random_state=7,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("Random Forest -> Melhor score:", rf_search.best_score_)
print("Random Forest -> Melhores params:", rf_search.best_params_)

### Gradient Boosting Refinado

In [ ]:
# --------------------------------------------------------------
# Gradient Boosting Refinado
# --------------------------------------------------------------
pipe_gb = Pipeline([('gb', GradientBoostingClassifier(random_state=7))])

gb_space = {
    'gb__n_estimators': [200, 400, 600],
    'gb__learning_rate': [0.05, 0.1, 0.2],
    'gb__max_depth': [2, 3, 4],
    'gb__subsample': [0.5, 0.7, 1.0],
}

gb_search = RandomizedSearchCV(
    estimator=pipe_gb,
    param_distributions=gb_space,
    n_iter=5,
    cv=kfold,
    scoring='accuracy',
    random_state=7,
    n_jobs=-1
)

gb_search.fit(X_train, y_train)

print("Gradient Boosting -> Melhor score:", gb_search.best_score_)
print("Gradient Boosting -> Melhores params:", gb_search.best_params_)

### Extra Trees refinado

In [ ]:
# --------------------------------------------------------------
# Extra Trees Refinado
# --------------------------------------------------------------
pipe_et = Pipeline([('et', ExtraTreesClassifier(random_state=7))])
et_space = {
    'et__n_estimators': [400, 800, 1200],
    'et__max_depth': [None, 20, 20],
    'et__max_features': ['sqrt', 0.5, 0.5],
    'et__min_samples_leaf': [1, 2, 2],
    'et__bootstrap': [False, False, False]  # ET costuma treinar full-sample
}
et_search = RandomizedSearchCV(pipe_et, et_space, n_iter=5, cv=kfold, scoring='accuracy', n_jobs=-1, random_state=7)
et_search.fit(X_train, y_train)
print("Gradient Boosting -> Melhor score:", et_search.best_score_)
print("Gradient Boosting -> Melhores params:", et_search.best_params_)

# Finalização do modelo

In [ ]:
# --- Avaliação no conjunto de teste (RF com os melhores hiperparâmetros) ---

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=7,
    n_jobs=-1
)

rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print("Acurácia (teste):", accuracy_score(y_test, pred))

In [ ]:
# --- Treino final com TODO o dataset (para produção) ---
rf_prod = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=7,
    n_jobs=-1
)
rf_prod.fit(X, y)

## Simulando a aplicação do modelo em dados não vistos

In [ ]:
# === Simulação de dados novos com RF ===
data = {
    'popularity': [96],
    'danceability': [0.704],
    'energy': [0.797],
    'loudness': [-5.927],
    'acousticness': [0.08260],
    'instrumentalness': [0.000745],
    'valence': [0.825],
    'tempo': [139.994],
}

atributos = [
    'popularity', 'danceability', 'energy', 'loudness',
    'acousticness', 'instrumentalness', 'valence', 'tempo'
]

entrada = pd.DataFrame(data, columns=atributos)

# Predição
pred = rf_prod.predict(entrada)[0]
proba = rf_prod.predict_proba(entrada)[0]

print("Classe prevista:", pred)
print("Probabilidades:", proba)


In [ ]:
# ---------- UI: dropdown artista | música (único por track_id) ----------
import ipywidgets as w
import pandas as pd
import numpy as np
from IPython.display import display

# === Ajuste estes nomes conforme o que você usou no treino ===
feature_cols = [
    'popularity','danceability','energy','loudness',
    'acousticness','instrumentalness','valence','tempo'
]

# modelo final (troque para gb_prod se for o seu)
model_final = rf_prod   # ou: gb_prod

# Opção única com id para evitar duplicados
df['_opt'] = df['artist_name'] + ' | ' + df['track_name'] + ' | ' + df['track_id']
opt2idx = dict(zip(df['_opt'], df.index))
dd = w.Dropdown(options=sorted(opt2idx.keys()), description='Música:', layout=w.Layout(width='70%'))

out = w.Output()

def on_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        idx = opt2idx[change['new']]
        row = df.loc[idx, feature_cols]
        entrada = row.to_frame().T  # 1xN, mesma ordem do treino

        pred = model_final.predict(entrada)[0]
        proba = model_final.predict_proba(entrada)[0]
        top = np.argsort(proba)[::-1][:3]

        with out:
            out.clear_output()
            print("🎵 Selecionado:", change['new'])
            print("\nFeatures usadas no modelo:")
            display(row.to_frame(name='valor'))
            print("\nClasse prevista:", pred)
            print("Top probabilidades:")
            for i in top:
                print(f" - {model_final.classes_[i]}: {proba[i]:.2%}")

dd.observe(on_change, names='value')
display(dd, out)

# Dispara uma predição inicial
on_change({'name':'value','new':dd.value})


# 10. Avaliação

O projeto atingiu seu objetivo de aplicar conceitos de aprendizado supervisionado para recomendar músicas de acordo com o contexto do usuário.

Foram testados diversos algoritmos de classificação, desde modelos lineares (Regressão Logística, SVM) até métodos de ensemble (Bagging, Random Forest, Extra Trees e Gradient Boosting).

Algumas das condições impostas e que não contribuiu para um melhor modelo, foi a qualidade dos dados. Percebeu-se que algumas faixas não pertenciam o gênero correto e que dificultou no tratamento dos dados, e algumas coisas acabaram passando.

Os resultados mostraram que os modelos baseados em ensembles tiveram desempenho superior, com destaque para o Random Forest.